# SignSync — MLP Training (Google Colab)

**No GPU needed.** Free Colab CPU runtime finishes in ~10–15 minutes.

Run each cell top-to-bottom. After training you will download:
- `sign_language_mlp.onnx` — runtime model (ONNX, CPU inference)
- `sign_language_mlp.pt`   — PyTorch weights (for fine-tuning later)
- `class_names_mlp.json`   — ordered class labels

**Architecture:** 63-D MediaPipe landmarks → FC(256) → BN → ReLU → Dropout(0.3) → FC(128) → BN → ReLU → Dropout(0.2) → FC(33) → Softmax  
**33 classes:** A-Z (26) + space + del + help + danger + emergency + thumbs_down + ok_sign

## Step 1 — Install dependencies

In [ ]:
!pip install -q torch torchvision mediapipe scikit-learn tqdm onnxruntime kaggle

## Step 2 — Kaggle credentials

Upload your `kaggle.json` (from https://www.kaggle.com/settings → API → Create New Token).
This gives access to the ASL Alphabet dataset (~1 GB, ~87 K images).

In [ ]:
from google.colab import files

uploaded = files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle credentials configured")

## Step 3 — Upload training script

Upload `train_standalone.py` from your project:
`src/app/core/ml/sign_model_mlp/train_standalone.py`

In [ ]:
from google.colab import files

files.upload()  # upload train_standalone.py
!ls -lh train_standalone.py

## Step 4 — Config (edit before running)

In [ ]:
# ── Edit these if needed ──────────────────────────────────────────────────
IMAGES_PER_LETTER = 500  # images per A-Z letter (500 → ~95% val accuracy)
SYNTHETIC_VARIATIONS = 500  # synthetic samples per emergency sign
EPOCHS = 60  # training epochs (60 is enough; more = diminishing returns)
LEARNING_RATE = 1e-3  # Adam learning rate
CONFIDENCE_THRESHOLD = 75  # % — predictions below this are "uncertain" in backend

import sys

sys.argv = [
    "train_standalone.py",
    "--images-per-letter",
    str(IMAGES_PER_LETTER),
    "--synthetic-variations",
    str(SYNTHETIC_VARIATIONS),
    "--epochs",
    str(EPOCHS),
    "--lr",
    str(LEARNING_RATE),
]

print("📋 Config:")
print(f"   Images/letter      : {IMAGES_PER_LETTER}")
print(f"   Synthetic/emergency: {SYNTHETIC_VARIATIONS}")
print(f"   Epochs             : {EPOCHS}")
print(f"   Learning rate      : {LEARNING_RATE}")
print(f"   Confidence threshold: {CONFIDENCE_THRESHOLD}%")
print()
print("ℹ️  Emergency signs (help/danger/emergency/thumbs_down/ok_sign)")
print("   use hand-crafted synthetic landmark templates — no real dataset needed.")
print("   They are augmented with gaussian noise + scale jitter for variety.")

## What's in this model

| Feature | Detail |
|---------|--------|
| Input | 63-D MediaPipe landmarks (21 pts × x,y,z — wrist-subtracted, scale-normalised) |
| Architecture | FC(256)→BN→ReLU→Drop(0.3) → FC(128)→BN→ReLU→Drop(0.2) → FC(33) |
| Parameters | ~85 K |
| Runtime | ONNX (no PyTorch at inference) — ~1 ms/frame on CPU |
| A-Z + space + del | Real landmarks extracted from Kaggle ASL alphabet dataset |
| Emergency signs | Synthetic: 500 augmented variations of hand-crafted templates |
| Optimizer | Adam (lr=1e-3, weight_decay=1e-4) |
| Scheduler | CosineAnnealingLR |
| Loss | CrossEntropyLoss with class weights (handles class imbalance) |

**Why synthetic for emergency signs?**  
No public dataset has these specific gestures in MediaPipe format. The synthetic approach
adds gaussian noise (σ=0.02) + scale jitter (±8%) to anatomically correct base poses — 
enough variation that the model generalises to real hands.

## Step 5 — Train!

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("mlp_train", "train_standalone.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
module.main()

## Step 6 — Download trained model

In [ ]:
import os

from google.colab import files

for f in [
    "trained_model/sign_language_mlp.onnx",
    "trained_model/sign_language_mlp.pt",
    "trained_model/class_names_mlp.json",
]:
    if os.path.exists(f):
        files.download(f)
        print(f"⬇️  Downloaded {f}")
    else:
        print(f"⚠️  Not found: {f}")

print()
print("📁 Place downloaded files in:")
print("   src/app/core/ml/sign_model_mlp/trained_model/")
print()
print("🔄 Then restart Docker:")
print("   docker compose restart web")
print("   (no rebuild needed — files are volume-mounted)")